In [9]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA modules
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.genereux_uncertainty_propagation as gup

importlib.reload(ep)
importlib.reload(em)
importlib.reload(gup)

# Define directories
data_dir = repo_dir / "Data/GrabSample_data"
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load entire RI series dataset
df_all = pd.read_csv(data_dir / "RI25-Hungerford-baseflow-variability.csv", low_memory=False)

# Load RI23 dataset for mixing space definition
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

# Target end-member categories
target_types = [
    "Baseflow",
    "Groundwater",
    "Snowmelt lysimeter",
    "Soil water lysimeter wet",
    "Soil water lysimeter dry",
    "Soil water lysimeter",
]

# Filter for 2024 (RI24-) and 2025 (RI25-) end-member samples for HUNGERFORD ONLY
em_raw_2425 = df_all[
    (df_all["Site"] == "Hungerford")
    & (
        #df_all["Sample ID"].str.startswith("RI24-", na=False)
        #| 
        df_all["Sample ID"].str.startswith("RI25-", na=False)
    )
    & (df_all["Type"].isin(target_types))
].copy()

# Print sample count summary by year prefix and end-member type
print(f"Total 2024–2025 Hungerford end-member samples filtered: {len(em_raw_2425)}\n")
print(
    em_raw_2425.groupby([em_raw_2425["Sample ID"].str[:4], "Type"]).size()
)

############################
# DEFINE RI23 Mixing space #
############################

Hungerford_tracers = ['Ca_mg_L', 'Cu_mg_L', 'Cl_mg_L', 'K_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']

# 1. Run EMMA function
(
    hungerford_entire_fractions_df,
    hungerford_entire_scaler,
    hungerford_entire_pca,
    hungerford_entire_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Hungerford",
    start_date="2023-01-01 00:00:00",
    end_date="2023-05-01 00:00:00",
    endmember_ids=[
        "RI23-1035", # Baseflow
        "RI23-5007", # SWLD
        "RI23-1061"  # Meltwater/Rain
    ]
)

# 2. Extract raw endmember rows 
em_raw_subset = df[df["Sample ID"].isin(["RI23-1035", "RI23-5007", "RI23-1061"])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Hungerford")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-01-01 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-05-01 00:00:00"))
].copy()

stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define laboratory tracer precisions [W(C_j)]
analytical_sd = {
    "Ca_mg_L": 0.05,   # mg/L
    "Cl_mg_L": 0.15,   # mg/L
    "Si_mg_L": 0.10,   # mg/L
    "K_mg_L": 0.10,    # mg/L
    "Cu_mg_L": 0.10,   # mg/L
    "Mg_mg_L": 0.02,   # mg/L
    "dD": 0.8,         # per mil
    "d18O": 0.08,      # per mil
    "Na_mg_L": 0.05    # mg/L
}

# Transform 2024-2025 Hungerford end-members into event PC-space
ordered_tracers = list(hungerford_entire_scaler.feature_names_in_)
em_clean = em_raw_2425.dropna(subset=ordered_tracers).copy()

em_scaled = hungerford_entire_scaler.transform(em_clean[ordered_tracers])
em_scores = hungerford_entire_pca.transform(em_scaled)[:, :2]

em_clean["PC1"] = em_scores[:, 0]
em_clean["PC2"] = em_scores[:, 1]

# Summary table of Hungerford end-member PC sample size and standard deviation
em_pc_stats = em_clean.groupby("Type")[["PC1", "PC2"]].agg(["count", "std"])
print("\n2024-2025 Hungerford End-Member Stats in PC Space:")
print(em_pc_stats)

Total 2024–2025 Hungerford end-member samples filtered: 2

Sample ID  Type    
RI25       Baseflow    2
dtype: int64

2024-2025 Hungerford End-Member Stats in PC Space:
           PC1            PC2         
         count      std count      std
Type                                  
Baseflow     2  0.69871     2  0.33859
